# Notebook 1: RAG Fundamentals & Basic Implementation

**Duration:** 90-120 minutes  
**Level:** Intermediate  

---

## Learning Objectives

By the end of this notebook, you will be able to:

- ✅ Understand core RAG architecture and components
- ✅ Implement a basic RAG pipeline using Haystack
- ✅ Work with document stores, retrievers, and generators
- ✅ Master prompt engineering for RAG applications
- ✅ Understand when to use RAG vs fine-tuning

---

## Table of Contents

1. [Introduction to RAG](#1-introduction-to-rag)
2. [Environment Setup](#2-environment-setup)
3. [Understanding RAG Components](#3-understanding-rag-components)
4. [Data Preparation](#4-data-preparation)
5. [Example 1: Simple Q&A System](#5-example-1-simple-qa-system)
6. [Example 2: Technical Documentation Assistant](#6-example-2-technical-documentation-assistant)
7. [Example 3: News Article Summarization with Citations](#7-example-3-news-article-summarization-with-citations)
8. [Prompt Engineering Best Practices](#8-prompt-engineering-best-practices)
9. [Basic Evaluation](#9-basic-evaluation)
10. [Exercises & Challenges](#10-exercises-challenges)
11. [Summary & Next Steps](#11-summary-next-steps)

---

## 1. Introduction to RAG

### What is RAG?

**Retrieval-Augmented Generation (RAG)** is an AI architecture that combines:
- **Large Language Models (LLMs)** for natural language understanding and generation
- **External knowledge retrieval** from document stores or databases

### Why RAG?

| Problem | RAG Solution |
|---------|-------------|
| LLMs have knowledge cutoff dates | Access to up-to-date information |
| LLMs can hallucinate facts | Grounded responses with source citations |
| Domain-specific knowledge needed | Connect to proprietary knowledge bases |
| Expensive to fine-tune for every update | Simply update document store |

### RAG vs Fine-Tuning

**Use RAG when:**
- Information changes frequently
- Need source citations/transparency
- Have document-based knowledge
- Want cost-effective updates

**Use Fine-Tuning when:**
- Need specific tone/style adaptation
- Require specialized reasoning patterns
- Have limited inference latency budget
- Knowledge is stable

### Basic RAG Architecture

```
User Query → Retriever → Relevant Docs → Prompt + Docs → LLM → Answer
                ↑
           Document Store
```

### Components in Haystack

1. **Document Store**: Storage for your documents
2. **Retriever**: Finds relevant documents for a query
3. **Prompt Builder**: Constructs prompts with retrieved context
4. **Generator**: LLM that generates the final answer
5. **Pipeline**: Orchestrates the flow between components

## 2. Environment Setup

First, let's import all necessary libraries and configure our environment.

In [1]:
# Standard library imports
import os
import json
import time
from typing import List, Dict, Any, Optional
from datetime import datetime

# Third-party imports
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Haystack imports
from haystack import Document, Pipeline
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.generators import OpenAIGenerator
from haystack.components.builders import PromptBuilder
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import ChatMessage
from haystack.utils import Secret

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

✅ All imports successful!


In [2]:
# Load environment variables
load_dotenv()

# Verify API key is set
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(
        "⚠️ OPENAI_API_KEY not found!\n"
        "Please create a .env file with your OpenAI API key.\n"
        "Example: OPENAI_API_KEY=sk-..."
    )

print("✅ Environment configured successfully!")
print(f"📍 API Key found: {OPENAI_API_KEY[:20]}...")

✅ Environment configured successfully!
📍 API Key found: sk-proj-DKrbGLW_syHN...


## 3. Understanding RAG Components

Let's explore each component in detail with simple examples.

### 3.1 Document Representation

In Haystack, documents are represented using the `Document` dataclass.

In [3]:
# Creating a simple document
doc = Document(
    content="Haystack is an open-source framework for building LLM applications.",
    meta={
        "source": "documentation",
        "category": "introduction",
        "date": "2025-01-01"
    }
)

print("Document ID:", doc.id)
print("Content:", doc.content)
print("Metadata:", doc.meta)

Document ID: 765db240323e56967d8da988e047efcdc3610cf8615e8f31dfb3eae69e1dea2a
Content: Haystack is an open-source framework for building LLM applications.
Metadata: {'source': 'documentation', 'category': 'introduction', 'date': '2025-01-01'}


### 3.2 Document Store

The Document Store is where all documents are stored. We'll use `InMemoryDocumentStore` for simplicity.

In [4]:
# Initialize an in-memory document store
document_store = InMemoryDocumentStore()

# Create sample documents
sample_docs = [
    Document(content="Python is a high-level programming language."),
    Document(content="Machine learning is a subset of artificial intelligence."),
    Document(content="RAG combines retrieval with generation for better LLM responses.")
]

# Write documents to store
document_store.write_documents(sample_docs)

print(f"✅ Stored {document_store.count_documents()} documents")
print("\nDocuments in store:")
for doc in document_store.filter_documents():
    print(f"  - {doc.content[:50]}...")

✅ Stored 3 documents

Documents in store:
  - Python is a high-level programming language....
  - Machine learning is a subset of artificial intelli...
  - RAG combines retrieval with generation for better ...


### 3.3 Retriever

The Retriever finds relevant documents based on a query. We'll use BM25 (keyword-based) retrieval.

In [5]:
# Initialize BM25 retriever
retriever = InMemoryBM25Retriever(document_store=document_store)

# Test retrieval
query = "What is RAG?"
result = retriever.run(query=query, top_k=2)

print(f"Query: '{query}'\n")
print(f"Retrieved {len(result['documents'])} documents:\n")
for i, doc in enumerate(result['documents'], 1):
    print(f"{i}. {doc.content}")
    print(f"   Score: {doc.score:.4f}\n")

Query: 'What is RAG?'

Retrieved 2 documents:

1. RAG combines retrieval with generation for better LLM responses.
   Score: 1.3718

2. Python is a high-level programming language.
   Score: 1.1878



### 3.4 Prompt Builder

The Prompt Builder constructs prompts by combining the query with retrieved documents.

In [6]:
# Define a prompt template
template = """
Answer the question based on the context provided.

Context:
{% for doc in documents %}
  {{ doc.content }}
{% endfor %}

Question: {{ query }}

Answer:
"""

prompt_builder = PromptBuilder(template=template)

# Build a prompt
prompt_result = prompt_builder.run(
    documents=result['documents'],
    query=query
)

print("Generated Prompt:")
print("=" * 50)
print(prompt_result['prompt'])
print("=" * 50)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Generated Prompt:

Answer the question based on the context provided.

Context:

  RAG combines retrieval with generation for better LLM responses.

  Python is a high-level programming language.


Question: What is RAG?

Answer:


### 3.5 Generator

The Generator (LLM) produces the final answer based on the prompt.

In [7]:
# Initialize OpenAI generator
generator = OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo",
    generation_kwargs={"temperature": 0.3}
)

# Generate answer
answer = generator.run(prompt=prompt_result['prompt'])

print("Generated Answer:")
print("=" * 50)
print(answer['replies'][0])
print("=" * 50)

Generated Answer:
RAG is a model that combines retrieval with generation for better LLM responses.


## 4. Data Preparation

Let's create helper functions to generate sample data for our examples.

In [8]:
def generate_company_knowledge_base() -> List[Document]:
    """
    Generate a synthetic company knowledge base with policies and information.
    
    Returns:
        List of Document objects representing company policies
    """
    
    company_data = [
        {
            "title": "Remote Work Policy",
            "content": """TechCorp allows employees to work remotely up to 3 days per week. 
            Remote work requests must be approved by direct managers. Employees must be available 
            during core hours (10 AM - 3 PM local time) and maintain productivity standards. 
            Equipment for remote work is provided by the company.""",
            "category": "HR",
            "last_updated": "2025-01-15"
        },
        {
            "title": "Vacation Policy",
            "content": """Full-time employees receive 20 days of paid vacation annually. 
            Vacation days accrue monthly (1.67 days per month). Employees must submit vacation 
            requests at least 2 weeks in advance through the HR portal. Unused vacation days 
            can be carried over up to 5 days into the next year.""",
            "category": "HR",
            "last_updated": "2025-01-10"
        },
        {
            "title": "Health Insurance",
            "content": """TechCorp offers comprehensive health insurance including medical, 
            dental, and vision coverage. The company covers 80% of premium costs for employees 
            and 50% for dependents. Insurance becomes effective on the first day of the month 
            following 30 days of employment.""",
            "category": "Benefits",
            "last_updated": "2025-01-20"
        },
        {
            "title": "Professional Development",
            "content": """Employees have access to $2,000 annually for professional development, 
            including conferences, courses, and certifications. Requests must be submitted through 
            the learning management system and approved by managers. Employees must remain with 
            the company for 12 months after using these funds or repay the amount.""",
            "category": "Learning",
            "last_updated": "2025-01-05"
        },
        {
            "title": "Code of Conduct",
            "content": """All employees must adhere to TechCorp's code of conduct, which includes 
            treating colleagues with respect, maintaining confidentiality, avoiding conflicts of 
            interest, and reporting unethical behavior. Violations may result in disciplinary 
            action up to and including termination.""",
            "category": "Compliance",
            "last_updated": "2024-12-01"
        },
        {
            "title": "Equipment and Technology",
            "content": """TechCorp provides all necessary equipment including laptops, monitors, 
            and software licenses. Employees can choose between Mac or Windows laptops. 
            Equipment is refreshed every 3 years. Personal use of company equipment is permitted 
            within reasonable limits.""",
            "category": "IT",
            "last_updated": "2025-01-12"
        },
        {
            "title": "Performance Reviews",
            "content": """Performance reviews are conducted bi-annually in January and July. 
            Reviews include self-assessment, peer feedback, and manager evaluation. Performance 
            ratings directly impact compensation adjustments and promotion eligibility. 
            Employees receive written feedback and development plans.""",
            "category": "HR",
            "last_updated": "2025-01-08"
        },
        {
            "title": "Security Protocols",
            "content": """All employees must use two-factor authentication for company systems. 
            Passwords must be at least 12 characters with complexity requirements. Employees 
            must not share credentials or leave workstations unlocked. Security incidents 
            must be reported immediately to the IT security team.""",
            "category": "Security",
            "last_updated": "2025-01-18"
        }
    ]
    
    documents = []
    for item in company_data:
        doc = Document(
            content=item["content"],
            meta={
                "title": item["title"],
                "category": item["category"],
                "last_updated": item["last_updated"],
                "source": "company_kb"
            }
        )
        documents.append(doc)
    
    return documents

# Generate the knowledge base
company_docs = generate_company_knowledge_base()
print(f"✅ Generated {len(company_docs)} company documents")
print("\nSample document:")
print(f"Title: {company_docs[0].meta['title']}")
print(f"Content: {company_docs[0].content[:100]}...")

✅ Generated 8 company documents

Sample document:
Title: Remote Work Policy
Content: TechCorp allows employees to work remotely up to 3 days per week. 
            Remote work requests ...


## 5. Example 1: Simple Q&A System

Let's build a complete RAG pipeline for answering questions about company policies.

### 5.1 Create Pipeline Components

In [9]:
# Initialize document store and add documents
company_store = InMemoryDocumentStore()
company_store.write_documents(company_docs)

print(f"📚 Document store contains {company_store.count_documents()} documents")

📚 Document store contains 8 documents


In [10]:
# Define the RAG prompt template
rag_template = """
You are a helpful HR assistant for TechCorp. Answer the employee's question based on the company policies provided.

Company Policies:
{% for doc in documents %}
---
Policy: {{ doc.meta.title }}
{{ doc.content }}
{% endfor %}

Employee Question: {{ query }}

Instructions:
- Provide a clear, concise answer based on the policies above
- If the information isn't in the policies, say so honestly
- Cite the specific policy title when applicable
- Be friendly and professional

Answer:
"""

### 5.2 Build the RAG Pipeline

Haystack uses a Pipeline to connect components together.

In [11]:
# Create pipeline components
retriever = InMemoryBM25Retriever(document_store=company_store)
prompt_builder = PromptBuilder(template=rag_template)
llm = OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo",
    generation_kwargs={"temperature": 0.2}  # Lower temperature for factual responses
)

# Build the pipeline
rag_pipeline = Pipeline()
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.add_component("llm", llm)

# Connect the components
rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder", "llm")

print("✅ RAG Pipeline created successfully!")

# Visualize the pipeline
print("\nPipeline structure:")
print(rag_pipeline)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✅ RAG Pipeline created successfully!

Pipeline structure:
🚅 Components
  - retriever: InMemoryBM25Retriever
  - prompt_builder: PromptBuilder
  - llm: OpenAIGenerator
🛤️ Connections
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.prompt (str)



### 5.3 Test the Pipeline

In [12]:
def ask_question(pipeline: Pipeline, question: str, top_k: int = 3) -> Dict[str, Any]:
      """
      Ask a question using the RAG pipeline.
      """
      start_time = time.time()

      result = pipeline.run({
          "retriever": {"query": question, "top_k": top_k},
          "prompt_builder": {"query": question}
      },
      include_outputs_from={"retriever", "prompt_builder", "llm"})

      end_time = time.time()

      # Debug: Print what keys are actually in the result
      print(f"DEBUG - Result keys: {result.keys()}")

      # Check if retriever output exists
      if "retriever" in result:
          retrieved_docs = result["retriever"]["documents"]
      else:
          print("WARNING: No retriever output found!")
          retrieved_docs = []

      return {
          "question": question,
          "answer": result["llm"]["replies"][0],
          "retrieved_docs": retrieved_docs,
          "response_time": end_time - start_time
      }

In [13]:
# def ask_question(pipeline: Pipeline, question: str, top_k: int = 3) -> Dict[str, Any]:
#     """
#     Ask a question using the RAG pipeline.
    
#     Args:
#         pipeline: The RAG pipeline
#         question: User's question
#         top_k: Number of documents to retrieve
        
#     Returns:
#         Dictionary containing answer and metadata
#     """
#     start_time = time.time()
    
#     result = pipeline.run({
#         "retriever": {"query": question, "top_k": top_k},
#         "prompt_builder": {"query": question}
#     })
    
#     end_time = time.time()
    
#     return {
#         "question": question,
#         "answer": result["llm"]["replies"][0],
#         "retrieved_docs": result["retriever"]["documents"],
#         "response_time": end_time - start_time
#     }

# Test with sample questions
test_questions = [
    "How many vacation days do I get per year?",
    "Can I work from home?",
    "What's the health insurance coverage?"
]

for question in test_questions:
    print("\n" + "="*80)
    print(f"❓ Question: {question}")
    print("="*80)
    
    result = ask_question(rag_pipeline, question)
    
    print(f"\n💡 Answer:\n{result['answer']}")
    print(f"\n⏱️  Response time: {result['response_time']:.2f}s")
    print(f"\n📄 Retrieved {len(result['retrieved_docs'])} relevant policies:")
    for i, doc in enumerate(result['retrieved_docs'], 1):
        print(f"   {i}. {doc.meta['title']} (score: {doc.score:.3f})")


❓ Question: How many vacation days do I get per year?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
As a full-time employee at TechCorp, you are entitled to receive 20 days of paid vacation annually. Vacation days accrue monthly at a rate of 1.67 days per month. You can carry over up to 5 unused vacation days into the next year. (Vacation Policy)

⏱️  Response time: 2.21s

📄 Retrieved 3 relevant policies:
   1. Vacation Policy (score: 8.576)
   2. Remote Work Policy (score: 4.856)
   3. Health Insurance (score: 4.193)

❓ Question: Can I work from home?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
Yes, you can work from home up to 3 days per week as per our Remote Work Policy. Please ensure to get approval from your direct manager and maintain productivity during core hours (10 AM - 3 PM local time). Equipment for remote work will be provided by the company as stated in the Equipment and Technology policy.

⏱️  Re

## 6. Example 2: Technical Documentation Assistant

Let's build a RAG system for answering Python programming questions using documentation.

In [14]:
def generate_python_docs() -> List[Document]:
    """
    Generate sample Python documentation snippets.
    """
    python_docs = [
        {
            "title": "List Comprehensions",
            "content": """List comprehensions provide a concise way to create lists. 
            Common applications are to make new lists where each element is the result of some 
            operations applied to each member of another sequence. 
            Syntax: [expression for item in iterable if condition]
            Example: squares = [x**2 for x in range(10)]""",
            "topic": "data_structures"
        },
        {
            "title": "Dictionaries",
            "content": """Dictionaries are unordered collections of key-value pairs. 
            Keys must be immutable (strings, numbers, tuples). Values can be any type.
            Creation: my_dict = {'key': 'value', 'number': 42}
            Access: my_dict['key'] or my_dict.get('key')
            Methods: keys(), values(), items(), update(), pop()""",
            "topic": "data_structures"
        },
        {
            "title": "Exception Handling",
            "content": """Use try-except blocks to handle exceptions gracefully.
            Syntax: try: risky_operation() except ExceptionType as e: handle_error(e)
            Multiple exceptions: except (TypeError, ValueError) as e:
            Finally block: Executes regardless of exception. Use for cleanup.
            Raise custom exceptions: raise ValueError('Invalid input')""",
            "topic": "error_handling"
        },
        {
            "title": "File I/O",
            "content": """Python provides built-in functions for file operations.
            Open file: with open('file.txt', 'r') as f: content = f.read()
            Modes: 'r' (read), 'w' (write), 'a' (append), 'rb' (read binary)
            Methods: read(), readline(), readlines(), write(), writelines()
            Context manager (with) automatically closes files.""",
            "topic": "io"
        },
        {
            "title": "Lambda Functions",
            "content": """Lambda functions are small anonymous functions defined with lambda keyword.
            Syntax: lambda arguments: expression
            Example: square = lambda x: x**2
            Common with map, filter, sorted: sorted(items, key=lambda x: x['value'])
            Limited to single expression, no statements.""",
            "topic": "functions"
        },
        {
            "title": "Generators",
            "content": """Generators are functions that return iterators using yield.
            Memory efficient for large sequences - values generated on-the-fly.
            Example: def count_up_to(n): for i in range(n): yield i
            Generator expressions: (x**2 for x in range(10))
            Use next() to get values or iterate in for loop.""",
            "topic": "iterators"
        }
    ]
    
    return [
        Document(
            content=doc["content"],
            meta={"title": doc["title"], "topic": doc["topic"], "source": "python_docs"}
        )
        for doc in python_docs
    ]

# Generate Python documentation
python_docs = generate_python_docs()
print(f"✅ Generated {len(python_docs)} Python documentation snippets")

✅ Generated 6 Python documentation snippets


In [15]:
# Create a new pipeline for Python documentation
python_store = InMemoryDocumentStore()
python_store.write_documents(python_docs)

# Template optimized for code documentation
code_template = """
You are a Python programming assistant. Help the user understand Python concepts based on the documentation.

Documentation:
{% for doc in documents %}
## {{ doc.meta.title }}
{{ doc.content }}

{% endfor %}

Question: {{ query }}

Provide a clear explanation with code examples where applicable.
Answer:
"""

# Build Python docs pipeline
python_pipeline = Pipeline()
python_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=python_store))
python_pipeline.add_component("prompt_builder", PromptBuilder(template=code_template))
python_pipeline.add_component("llm", OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo"
))

python_pipeline.connect("retriever.documents", "prompt_builder.documents")
python_pipeline.connect("prompt_builder", "llm")

print("✅ Python documentation pipeline ready!")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✅ Python documentation pipeline ready!


In [16]:
# Test Python documentation assistant
python_questions = [
    "How do I handle errors in Python?",
    "What's the difference between a list comprehension and a generator?",
    "How do I read a file safely?"
]

for question in python_questions:
    print("\n" + "="*80)
    print(f"❓ {question}")
    print("="*80)
    
    result = ask_question(python_pipeline, question, top_k=2)
    print(f"\n{result['answer']}")


❓ How do I handle errors in Python?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

In Python, you can handle errors using try-except blocks. This allows you to gracefully handle exceptions that may occur during the execution of your code. Here is an example of how you can use try-except blocks:

```python
def divide_numbers(num1, num2):
    try:
        result = num1 / num2
        print("Result:", result)
    except ZeroDivisionError as e:
        print("Error: Cannot divide by zero")
    except TypeError as e:
        print("Error: Invalid input types")
    except Exception as e:
        print("An error occurred:", e)

# Example usage
divide_numbers(10, 2)  # Result: 5.0
divide_numbers(10, 0)  # Error: Cannot divide by zero
divide_numbers('10', 2)  # Error: Invalid input types
divide_numbers(10, '2')  # Error: Invalid input types
```

In the above code, we have a `divide_numbers` function that attempts to divide two numbers. We use a try-except block to catc

## 7. Example 3: News Article Summarization with Citations

Let's build a system that answers questions about news articles with proper source citations.

In [17]:
def generate_news_articles() -> List[Document]:
    """
    Generate sample news articles.
    """
    news_data = [
        {
            "title": "Major Breakthrough in Quantum Computing",
            "content": """Scientists at Tech University announced a significant breakthrough in 
            quantum computing, achieving quantum supremacy with a 1000-qubit processor. The new 
            system can solve complex optimization problems 1000x faster than classical supercomputers. 
            Lead researcher Dr. Smith states this could revolutionize drug discovery and climate modeling.""",
            "category": "Technology",
            "date": "2025-01-28",
            "source_url": "https://technews.example.com/quantum-breakthrough"
        },
        {
            "title": "New Climate Agreement Reached",
            "content": """World leaders agreed to reduce carbon emissions by 50% by 2035 at the 
            Global Climate Summit. The agreement includes $500 billion in funding for renewable energy 
            in developing nations. Environmental groups praised the commitment while noting challenges 
            in implementation.""",
            "category": "Environment",
            "date": "2025-01-27",
            "source_url": "https://worldnews.example.com/climate-agreement"
        },
        {
            "title": "AI Assists in Medical Diagnosis",
            "content": """A new AI system developed by MedTech Corp can diagnose rare diseases 
            with 95% accuracy from medical imaging. The system has been trained on 10 million medical 
            images and can identify patterns invisible to human doctors. Clinical trials show 30% 
            improvement in early detection rates.""",
            "category": "Healthcare",
            "date": "2025-01-26",
            "source_url": "https://healthnews.example.com/ai-diagnosis"
        },
        {
            "title": "Electric Vehicle Sales Surge",
            "content": """Electric vehicle sales increased 200% year-over-year, now comprising 
            25% of all new car sales globally. Major automakers announced plans to phase out combustion 
            engines by 2030. Battery technology improvements have extended range to 500+ miles while 
            reducing costs by 40%.""",
            "category": "Automotive",
            "date": "2025-01-25",
            "source_url": "https://autonews.example.com/ev-sales-surge"
        }
    ]
    
    return [
        Document(
            content=doc["content"],
            meta={
                "title": doc["title"],
                "category": doc["category"],
                "date": doc["date"],
                "source_url": doc["source_url"],
                "source": "news"
            }
        )
        for doc in news_data
    ]

news_docs = generate_news_articles()
print(f"✅ Generated {len(news_docs)} news articles")

✅ Generated 4 news articles


In [18]:
# Template with citation requirements
news_template = """
You are a news assistant. Answer questions based on the news articles provided.

News Articles:
{% for doc in documents %}
---
Title: {{ doc.meta.title }}
Date: {{ doc.meta.date }}
Source: {{ doc.meta.source_url }}
Content: {{ doc.content }}
{% endfor %}

Question: {{ query }}

Instructions:
- Answer based on the articles provided
- Include citations with article titles and dates
- If information is not in the articles, state that clearly
- Provide source URLs when relevant

Answer:
"""

# Build news pipeline
news_store = InMemoryDocumentStore()
news_store.write_documents(news_docs)

news_pipeline = Pipeline()
news_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=news_store))
news_pipeline.add_component("prompt_builder", PromptBuilder(template=news_template))
news_pipeline.add_component("llm", OpenAIGenerator(
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    model="gpt-3.5-turbo"
))

news_pipeline.connect("retriever.documents", "prompt_builder.documents")
news_pipeline.connect("prompt_builder", "llm")

print("✅ News pipeline ready!")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✅ News pipeline ready!


In [19]:
# Test with news questions
news_questions = [
    "What are the recent developments in AI for healthcare?",
    "What progress has been made in quantum computing?",
    "What's happening with electric vehicles?"
]

for question in news_questions:
    print("\n" + "="*80)
    print(f"❓ {question}")
    print("="*80)
    
    result = ask_question(news_pipeline, question, top_k=2)
    print(f"\n{result['answer']}")
    
    print("\n📰 Sources:")
    for doc in result['retrieved_docs']:
        print(f"  • {doc.meta['title']} ({doc.meta['date']})")
        print(f"    {doc.meta['source_url']}")


❓ What are the recent developments in AI for healthcare?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

The recent development in AI for healthcare is a new AI system developed by MedTech Corp, as reported in the article "AI Assists in Medical Diagnosis" dated 2025-01-26. This AI system can diagnose rare diseases with 95% accuracy from medical imaging and has been trained on 10 million medical images. Clinical trials have shown a 30% improvement in early detection rates due to this AI system. Source: https://healthnews.example.com/ai-diagnosis.

📰 Sources:
  • New Climate Agreement Reached (2025-01-27)
    https://worldnews.example.com/climate-agreement
  • AI Assists in Medical Diagnosis (2025-01-26)
    https://healthnews.example.com/ai-diagnosis

❓ What progress has been made in quantum computing?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

Scientists at Tech University announced a major breakthrough in quantum computing, achievi

## 8. Prompt Engineering Best Practices

Let's explore how different prompt structures affect RAG performance.

In [20]:
# Different prompt templates to compare
prompt_variations = {
    "basic": """
Context: {% for doc in documents %}{{ doc.content }}{% endfor %}
Question: {{ query }}
Answer:
""",
    
    "structured": """
You are a helpful assistant. Use the context below to answer the question.

CONTEXT:
{% for doc in documents %}
[{{ loop.index }}] {{ doc.content }}
{% endfor %}

QUESTION: {{ query }}

ANSWER:
""",
    
    "with_instructions": """
You are a precise assistant. Follow these guidelines:
1. Base your answer ONLY on the context provided
2. If the context doesn't contain the answer, say "I don't have enough information"
3. Be concise but complete
4. Cite specific parts of the context when possible

CONTEXT:
{% for doc in documents %}
Source {{ loop.index }}: {{ doc.content }}
{% endfor %}

USER QUESTION: {{ query }}

YOUR ANSWER:
""",
    
    "chain_of_thought": """
You are a thoughtful assistant. Answer the question step-by-step.

CONTEXT:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

QUESTION: {{ query }}

Let's think through this step by step:
1. First, identify relevant information from the context
2. Then, synthesize the information
3. Finally, provide a clear answer

ANSWER:
"""
}

print("✅ Created 4 prompt variations for comparison")

✅ Created 4 prompt variations for comparison


In [21]:
def compare_prompts(question: str, store: InMemoryDocumentStore, templates: Dict[str, str]):
    """
    Compare different prompt templates for the same question.
    """
    print(f"\n{'='*80}")
    print(f"Comparing prompts for: {question}")
    print(f"{'='*80}\n")
    
    for name, template in templates.items():
        # Build temporary pipeline with this template
        temp_pipeline = Pipeline()
        temp_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=store))
        temp_pipeline.add_component("prompt_builder", PromptBuilder(template=template))
        temp_pipeline.add_component("llm", OpenAIGenerator(
            api_key=Secret.from_env_var("OPENAI_API_KEY"),
            model="gpt-3.5-turbo"
        ))
        temp_pipeline.connect("retriever.documents", "prompt_builder.documents")
        temp_pipeline.connect("prompt_builder", "llm")
        
        result = ask_question(temp_pipeline, question, top_k=2)
        
        print(f"\n📝 Template: {name.upper()}")
        print("-" * 80)
        print(result['answer'])
        print(f"\n⏱️ Response time: {result['response_time']:.2f}s")

# Compare prompts
compare_prompts(
    "How many vacation days do employees get?",
    company_store,
    prompt_variations
)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Comparing prompts for: How many vacation days do employees get?



PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: BASIC
--------------------------------------------------------------------------------
Full-time employees receive 20 days of paid vacation annually.

⏱️ Response time: 0.59s


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: STRUCTURED
--------------------------------------------------------------------------------
Employees get 20 days of paid vacation annually.

⏱️ Response time: 0.60s


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: WITH_INSTRUCTIONS
--------------------------------------------------------------------------------
Full-time employees receive 20 days of paid vacation annually, which accrue monthly at a rate of 1.67 days per month. Unused vacation days can be carried over up to 5 days into the next year.

⏱️ Response time: 1.98s
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: CHAIN_OF_THOUGHT
--------------------------------------------------------------------------------
1. Relevant information:
- Full-time employees receive 20 days of paid vacation annually
- Vacation days accrue monthly at a rate of 1.67 days per month
- Unused vacation days can be carried over up to 5 days into the next year

2. Synthesis:
To calculate the total number of vacation days employees get:
20 days/year + 5 carry-over days = 25 total vacation days possible per year
1.67 days/month x 12 months = 20.04 d

### Key Prompt Engineering Insights

1. **Clear Instructions**: Explicit guidelines improve answer quality
2. **Structure**: Well-formatted context helps LLM parse information
3. **Constraints**: Tell the model what NOT to do (e.g., don't make up info)
4. **Chain of Thought**: Asking for step-by-step reasoning can improve accuracy
5. **Citation Requirements**: Explicitly ask for sources if needed

## 9. Basic Evaluation

Let's implement simple evaluation metrics for our RAG system.

In [28]:
def evaluate_retrieval_precision(pipeline: Pipeline, test_cases: List[Dict[str, Any]]):
    """
    Evaluate retrieval precision - are retrieved documents relevant?
    
    Args:
        pipeline: RAG pipeline to evaluate
        test_cases: List of dicts with 'query' and 'expected_doc_titles'
    """
    results = []
    
    for case in test_cases:
        query = case['query']
        expected_titles = set(case['expected_doc_titles'])
        
        result = ask_question(pipeline, query, top_k=3)
        retrieved_titles = {doc.meta['title'] for doc in result['retrieved_docs']}
        
        # Calculate precision: relevant retrieved / total retrieved
        relevant_retrieved = retrieved_titles.intersection(expected_titles)
        precision = len(relevant_retrieved) / len(retrieved_titles) if retrieved_titles else 0
        
        results.append({
            'query': query,
            'precision': precision,
            'retrieved': list(retrieved_titles),
            'expected': list(expected_titles),
            'relevant_found': list(relevant_retrieved)
        })
    
    # Calculate average precision
    avg_precision = np.mean([r['precision'] for r in results])
    
    return results, avg_precision

# Define test cases
test_cases = [
    {
        'query': 'How many vacation days do I get?',
        'expected_doc_titles': ['Vacation Policy']
    },
    {
        'query': 'Can I work remotely?',
        'expected_doc_titles': ['Remote Work Policy']
    },
    {
        'query': 'What health benefits are available?',
        'expected_doc_titles': ['Health Insurance']
    },
    {
        'query': 'How does performance review work?',
        'expected_doc_titles': ['Performance Reviews']
    }
]

# Evaluate
results, avg_precision = evaluate_retrieval_precision(rag_pipeline, test_cases)

print("\n" + "="*80)
print("RETRIEVAL PRECISION EVALUATION")
print("="*80)

for result in results:
    print(f"\nQuery: {result['query']}")
    print(f"Precision: {result['precision']:.2%}")
    print(f"Expected: {result['expected']}")
    print(f"Retrieved: {result['retrieved']}")
    print(f"Relevant found: {result['relevant_found']}")

print(f"\n{'='*80}")
print(f"Average Precision: {avg_precision:.2%}")
print(f"{'='*80}")

DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

RETRIEVAL PRECISION EVALUATION

Query: How many vacation days do I get?
Precision: 33.33%
Expected: ['Vacation Policy']
Retrieved: ['Vacation Policy', 'Remote Work Policy', 'Health Insurance']
Relevant found: ['Vacation Policy']

Query: Can I work remotely?
Precision: 33.33%
Expected: ['Remote Work Policy']
Retrieved: ['Vacation Policy', 'Equipment and Technology', 'Remote Work Policy']
Relevant found: ['Remote Work Policy']

Query: What health benefits are available?
Precision: 33.33%
Expected: ['Health Insurance']
Retrieved: ['Performance Reviews', 'Remote Work Policy', 'Health Insurance']
Relevant found: ['Health Insurance']

Query: How does performance review work?
Precision: 33.33%
Expected: ['Perform

In [29]:
def measure_response_times(pipeline: Pipeline, queries: List[str], num_runs: int = 3):
    """
    Measure and analyze response times.
    """
    times = []
    
    for query in queries:
        query_times = []
        for _ in range(num_runs):
            result = ask_question(pipeline, query)
            query_times.append(result['response_time'])
        times.append({
            'query': query,
            'mean': np.mean(query_times),
            'std': np.std(query_times),
            'min': np.min(query_times),
            'max': np.max(query_times)
        })
    
    return times

# Measure response times
timing_queries = [
    "How many vacation days?",
    "What's the remote work policy?",
    "Health insurance coverage?"
]

timings = measure_response_times(rag_pipeline, timing_queries, num_runs=2)

print("\n" + "="*80)
print("RESPONSE TIME ANALYSIS")
print("="*80)

timing_df = pd.DataFrame(timings)
print("\n", timing_df.to_string(index=False))

print(f"\nOverall Average: {timing_df['mean'].mean():.2f}s")
print(f"Overall Std Dev: {timing_df['std'].mean():.2f}s")

DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

RESPONSE TIME ANALYSIS

                          query     mean      std      min      max
       How many vacation days? 1.751844 0.559173 1.192671 2.311017
What's the remote work policy? 0.922840 0.111756 0.811084 1.034597
    Health insurance coverage? 0.941337 0.150258 0.791079 1.091594

Overall Average: 1.21s
Overall Std Dev: 0.27s


## 10. Exercises & Challenges

Try these exercises to reinforce your learning:

### Exercise 1: Add More Documents

Extend the company knowledge base with 3-5 additional policies:
- Parental leave policy
- Referral bonus program
- Office amenities
- Relocation assistance

Test if the RAG system retrieves them correctly.

In [ ]:
# Your code here
# TODO: Create new documents and add to document store
# TODO: Test with relevant queries

### Exercise 2: Improve Prompt Template

Create a prompt template that:
1. Asks the LLM to rate its confidence (1-10)
2. Provides both a short answer and detailed explanation
3. Lists which documents were most useful

Compare results with the basic template.

In [ ]:
# Your code here
# TODO: Design improved template
# TODO: Create pipeline with new template
# TODO: Compare with basic pipeline

### Exercise 3: Handle Out-of-Scope Questions

Test the pipeline with questions that cannot be answered from the knowledge base:
- "What's the weather today?"
- "How do I cook pasta?"
- "What's the stock price?"

Does the system appropriately say it doesn't know? How can you improve this?

In [ ]:
# Your code here
# TODO: Test with out-of-scope questions
# TODO: Analyze responses
# TODO: Modify prompt to handle better

### Challenge: Build a Multi-Topic RAG System

Combine the company policies, Python documentation, and news articles into a single system that:
1. Automatically determines which knowledge base to search
2. Routes queries appropriately
3. Provides answers with source attribution

Hint: You might need multiple retrievers or metadata filtering.

In [ ]:
# Your code here
# TODO: Combine all documents
# TODO: Add topic/category metadata
# TODO: Implement query routing logic
# TODO: Build unified pipeline

## 11. Summary & Next Steps

### What You've Learned

✅ **RAG Fundamentals**
- What RAG is and when to use it
- Core components: Document Store, Retriever, Prompt Builder, Generator
- How components connect in a pipeline

✅ **Practical Implementation**
- Built 3 different RAG systems (HR, documentation, news)
- Worked with real-world use cases
- Handled different document types and structures

✅ **Prompt Engineering**
- Compared different prompt structures
- Learned best practices for RAG prompts
- Understood importance of clear instructions

✅ **Evaluation**
- Measured retrieval precision
- Analyzed response times
- Basic performance metrics

### Limitations of Basic RAG

The approach we've used has some limitations:

1. **Keyword-based retrieval (BM25)** misses semantic similarity
2. **No reranking** - first retrieved docs might not be most relevant
3. **Static pipeline** - no adaptive behavior
4. **No conversation memory** - each query is independent
5. **Limited evaluation** - we haven't measured answer quality systematically

### Next Steps

In **Notebook 2 - Advanced Retrieval Techniques**, you'll learn:

- 🔍 **Semantic search** with dense embeddings
- 🔄 **Hybrid retrieval** (BM25 + embeddings)
- 📈 **Reranking** for improved precision
- 🎯 **Query expansion** for better recall
- 📊 **Comprehensive evaluation** metrics

### Additional Resources

- [Haystack Documentation](https://docs.haystack.deepset.ai/)
- [RAG Paper (Lewis et al.)](https://arxiv.org/abs/2005.11401)
- [Prompt Engineering Guide](https://www.promptingguide.ai/)

---

**Congratulations!** You've completed the RAG Fundamentals notebook. You now have a solid foundation for building more sophisticated RAG systems.

Ready to level up? Continue to **Notebook 2: Advanced Retrieval Techniques**! 🚀

---

## Appendix: Helper Functions Reference

Here's a quick reference of all helper functions created in this notebook:

In [ ]:
# Helper functions summary
helper_functions = {
    "generate_company_knowledge_base()": "Creates synthetic company policy documents",
    "generate_python_docs()": "Creates Python documentation snippets",
    "generate_news_articles()": "Creates sample news articles",
    "ask_question(pipeline, question, top_k)": "Runs a query through RAG pipeline",
    "evaluate_retrieval_precision(pipeline, test_cases)": "Evaluates retrieval quality",
    "measure_response_times(pipeline, queries, num_runs)": "Measures pipeline latency",
    "compare_prompts(question, store, templates)": "Compares different prompt templates"
}

print("Helper Functions:")
for func, description in helper_functions.items():
    print(f"  • {func}: {description}")